In [ ]:
# In[1]:

import pandas as pd

# Read files with timestamp parsed as UTC datetimes (unit='s', utc=True)
df_metric = pd.read_csv('metric.csv')
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

df_trace = pd.read_csv('trace.csv')
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

df_log = pd.read_csv('log.csv')
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

df_error = pd.read_csv('error_logs.csv')
df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# ------- metric.csv summaries -------
# 1) distinct cmdb_id values and counts (top 20)
metric_cmdb_counts = (
    df_metric.groupby('cmdb_id')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# 2) distinct kpi_name values and counts (top 20)
metric_kpi_counts = (
    df_metric.groupby('kpi_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# 3) For each (cmdb_id, kpi_name) compute count, global P95 of 'value', min/max timestamp; top 20 by count
metric_pairs = (
    df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)
    .agg(
        count=('value', 'size'),
        p95_value=('value', lambda x: x.quantile(0.95)),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
    )
    .sort_values('count', ascending=False)
    .head(20)
)

# ------- trace.csv summaries -------
trace_cmdb_counts = (
    df_trace.groupby('cmdb_id')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

trace_name_counts = (
    df_trace.groupby('trace_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

trace_pairs = (
    df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)
    .agg(
        count=('value', 'size'),
        p95_value=('value', lambda x: x.quantile(0.95)),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
    )
    .sort_values('count', ascending=False)
    .head(20)
)

# ------- log.csv summaries -------
log_cmdb_counts = (
    df_log.groupby('cmdb_id')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

log_name_counts = (
    df_log.groupby('log_name')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

log_pairs = (
    df_log.groupby(['cmdb_id', 'log_name'], as_index=False)
    .agg(
        count=('value', 'size'),
        p95_value=('value', lambda x: x.quantile(0.95)),
        min_timestamp=('timestamp', 'min'),
        max_timestamp=('timestamp', 'max'),
    )
    .sort_values('count', ascending=False)
    .head(20)
)

# ------- error_logs.csv summaries -------
# distinct cmdb_id values (counts)
error_cmdb_list_counts = (
    df_error.groupby('cmdb_id')
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# up to 10 sample rows ordered by timestamp ascending (compact)
error_samples = df_error.sort_values('timestamp').head(10)[['timestamp', 'cmdb_id', 'message']]

# Display compact aggregated results (each variable is limited to <=20 rows)
metric_cmdb_counts, metric_kpi_counts, metric_pairs, trace_cmdb_counts, trace_name_counts, trace_pairs, log_cmdb_counts, log_name_counts, log_pairs, error_cmdb_list_counts, error_samples

```
Out[1]:
```
Summary (plain English, timestamps in UTC):

- metric.csv
  - Top cmdb_ids by row count include ts-admin-basic-info-service, ts-admin-travel-service, ts-ui-dashboard, ts-travel-service (about 200 rows each); many services around 175–200 rows.
  - Top KPI names overall: cpu (1700 rows), mem (1700), socket (1700), diskio (1675), latency-90 (700), latency-50 (700), workload (700), error (123).
  - Per (cmdb_id, kpi) pairs: most reported pairs in the top list have 25 samples each, covering the time window 2024-02-20 18:57:00 → 2024-02-20 19:21:00 UTC. Examples of P95 values: ts-admin-basic-info-service mem p95 ≈ 2.33e8, cpu p95 ≈ 0.768, diskio p95 ≈ 5.74e4; some services (e.g., ts-travel2-service) also show very large mem and socket P95s.

- trace.csv
  - Top cmdb_ids by row count: root (1352), ts-preserve-service (1232), ts-preserve-other-service (1216), ts-travel-service (1152), etc.
  - Common trace names: many trace.from_root.* metrics (duration_mean, duration_p95, error_rate, row_count) are frequent (counts ~338).
  - Per (cmdb_id, trace_name) pairs in the top results mostly have 24 samples covering 2024-02-20 18:57:00 → 2024-02-20 19:20:00 UTC. Notable P95s: trace.to_ts-admin-travel-service.duration_p95 ≈ 1.321s, trace.to_ts-auth-service.duration_p95 ≈ 1.108s, trace.to_ts-travel-service.duration_p95 ≈ 0.748s. Row_count p95 values show which targets received heavier traffic (examples: to ts-order-other-service row_count p95 large for ts-travel2-service).

- log.csv
  - Top cmdb_ids (many services) with 48 rows each in the top list.
  - Log names: log.error_count (1080 rows) and log.row_count (1080 rows).
  - Per (cmdb_id, log_name) pairs: top pairs have 24 samples between 2024-02-20 18:57:00 and 2024-02-20 19:20:00 UTC. log.error_count P95 is mostly 0 for many services; log.row_count P95 is large for some services (examples: ts-basic-service row_count p95 ≈ 3041.65, ts-auth-service ≈ 392.8, ts-admin-travel-service ≈ 212.35).

- error_logs.csv
  - No error-log sample rows were returned (empty result), so no example messages available.

Overall: telemetry covers a focused short window around 2024-02-20 18:57–19:21 UTC. Metrics and traces show elevated P95s for some memory/disk/socket KPIs and several service-to-service trace P95s near or above 1s (notably admin-travel and auth); log.error_count is generally zero while log.row_count indicates high request/row volumes for certain services.

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id  count
0   ts-admin-basic-info-service    200
3       ts-admin-travel-service    200
62              ts-ui-dashboard    200
59            ts-travel-service    200
47              ts-seat-service    188
25              ts-food-service    185
39    ts-preserve-other-service    175
42             ts-price-service    175
16     ts-consign-price-service    175
6          ts-assurance-service    175
10             ts-basic-service    175
8               ts-auth-service    175
34             ts-order-service    175
27    ts-inside-payment-service    175
22          ts-food-map-service    175
33       ts-order-other-service    175
36           ts-payment-service    175
17           ts-consign-service    175
19          ts-contacts-service    175
13            ts-config-service    175,      kpi_name  count
0         cpu   1700
5         mem   1700
6      socket   1700
1      diskio   1675
4  latency-90    700
3  latency-50    700
7    workload    700
2       error    123,                          cmdb_id    kpi_name  count     p95_value             min_timestamp             max_timestamp
360           ts-voucher-service      socket     25  4.000000e+00 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
0    ts-admin-basic-info-service         cpu     25  7.678605e-01 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
1    ts-admin-basic-info-service      diskio     25  5.738679e+04 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
2    ts-admin-basic-info-service       error     25  0.000000e+00 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
3    ts-admin-basic-info-service  latency-50     25  1.756127e-02 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
4    ts-admin-basic-info-service  latency-90     25  2.361029e-02 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
5    ts-admin-basic-info-service         mem     25  2.334751e+08 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
6    ts-admin-basic-info-service      socket     25  7.816667e+00 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
7    ts-admin-basic-info-service    workload     25  6.970700e-01 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
321             ts-travel2-mongo         mem     25  6.804068e+07 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
322             ts-travel2-mongo      socket     25  8.000000e+00 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
323           ts-travel2-service         cpu     25  5.639477e+00 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
324           ts-travel2-service      diskio     25  5.795640e+04 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
325           ts-travel2-service  latency-50     25  1.911473e-02 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
326           ts-travel2-service  latency-90     25  3.780345e-01 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
327           ts-travel2-service         mem     25  2.745353e+08 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
328           ts-travel2-service      socket     25  2.110667e+01 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
313            ts-travel-service       error     25  2.266000e-01 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
314            ts-travel-service  latency-50     25  2.102240e-02 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00
315            ts-travel-service  latency-90     25  7.479956e-01 2024-02-20 18:57:00+00:00 2024-02-20 19:21:00+00:00,                         cmdb_id  count
0                          root   1352
17          ts-preserve-service   1232
16    ts-preserve-other-service   1216
25            ts-travel-service   1152
20              ts-seat-service   1060
26           ts-travel2-service   1056
13       ts-order-other-service    860
14             ts-order-service    860
22           ts-station-service    768
11              ts-food-service    748
5              ts-basic-service    676
23        ts-ticketinfo-service    672
21          ts-security-service    576
12    ts-inside-payment-service    564
1   ts-admin-basic-info-service    480
19             ts-route-service    480
24             ts-train-service    480
9           ts-contacts-service    480
2       ts-admin-travel-service    480
3          ts-assurance-service    436,                                             trace_name  count
0                        trace.from_root.duration_mean    338
1                         trace.from_root.duration_p95    338
2                           trace.from_root.error_rate    338
3                            trace.from_root.row_count    338
71            trace.from_ts-preserve-service.row_count    260
70           trace.from_ts-preserve-service.error_rate    260
69         trace.from_ts-preserve-service.duration_p95    260
68        trace.from_ts-preserve-service.duration_mean    260
67      trace.from_ts-preserve-other-service.row_count    256
66     trace.from_ts-preserve-other-service.error_rate    256
65   trace.from_ts-preserve-other-service.duration_p95    256
64   trace.from_ts-preserve-other-service.duration_...    256
196          trace.to_ts-station-service.duration_mean    168
197           trace.to_ts-station-service.duration_p95    168
198             trace.to_ts-station-service.error_rate    168
199              trace.to_ts-station-service.row_count    168
163          trace.to_ts-order-other-service.row_count    167
162         trace.to_ts-order-other-service.error_rate    167
161       trace.to_ts-order-other-service.duration_p95    167
160      trace.to_ts-order-other-service.duration_mean    167,                 cmdb_id                                         trace_name  count  p95_value             min_timestamp             max_timestamp
783     ts-user-service                 trace.to_ts-user-service.row_count     24  21.700000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
0                  root  trace.to_ts-admin-basic-info-service.duration_...     24   0.011730 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
1                  root  trace.to_ts-admin-basic-info-service.duration_p95     24   0.014861 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
2                  root    trace.to_ts-admin-basic-info-service.error_rate     24   0.000000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
3                  root     trace.to_ts-admin-basic-info-service.row_count     24  21.850000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
4                  root     trace.to_ts-admin-travel-service.duration_mean     24   0.607907 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
5                  root      trace.to_ts-admin-travel-service.duration_p95     24   1.321222 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
6                  root        trace.to_ts-admin-travel-service.error_rate     24   0.000000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
7                  root         trace.to_ts-admin-travel-service.row_count     24  11.000000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
8                  root        trace.to_ts-assurance-service.duration_mean     24   0.003899 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
9                  root         trace.to_ts-assurance-service.duration_p95     24   0.004733 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
10                 root           trace.to_ts-assurance-service.error_rate     24   0.000000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
11                 root            trace.to_ts-assurance-service.row_count     24  10.850000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
12                 root             trace.to_ts-auth-service.duration_mean     24   0.441304 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
13                 root              trace.to_ts-auth-service.duration_p95     24   1.107602 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
14                 root                trace.to_ts-auth-service.error_rate     24   0.000000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
744  ts-travel2-service      trace.to_ts-order-other-service.duration_mean     24   0.007430 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
745  ts-travel2-service       trace.to_ts-order-other-service.duration_p95     24   0.014280 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
746  ts-travel2-service         trace.to_ts-order-other-service.error_rate     24   0.000000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
747  ts-travel2-service          trace.to_ts-order-other-service.row_count     24  93.700000 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00,                         cmdb_id  count
0   ts-admin-basic-info-service     48
1       ts-admin-travel-service     48
2            ts-assurance-mongo     48
3          ts-assurance-service     48
4                 ts-auth-mongo     48
5               ts-auth-service     48
6              ts-basic-service     48
7               ts-config-mongo     48
8              ts-consign-mongo     48
9        ts-consign-price-mongo     48
10           ts-consign-service     48
11            ts-contacts-mongo     48
12          ts-contacts-service     48
13            ts-food-map-mongo     48
14          ts-food-map-service     48
15                ts-food-mongo     48
16              ts-food-service     48
17      ts-inside-payment-mongo     48
18    ts-inside-payment-service     48
19        ts-notification-mongo     48,           log_name  count
0  log.error_count   1080
1    log.row_count   1080,                         cmdb_id         log_name  count  p95_value             min_timestamp             max_timestamp
0   ts-admin-basic-info-service  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
1   ts-admin-basic-info-service    log.row_count     24      32.70 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
2       ts-admin-travel-service  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
3       ts-admin-travel-service    log.row_count     24     212.35 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
4            ts-assurance-mongo  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
5            ts-assurance-mongo    log.row_count     24       1.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
6          ts-assurance-service  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
7          ts-assurance-service    log.row_count     24      15.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
8                 ts-auth-mongo  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
9                 ts-auth-mongo    log.row_count     24       1.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
10              ts-auth-service  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
11              ts-auth-service    log.row_count     24     392.80 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
12             ts-basic-service  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
13             ts-basic-service    log.row_count     24    3041.65 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
14              ts-config-mongo  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
15              ts-config-mongo    log.row_count     24       1.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
16             ts-consign-mongo  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
17             ts-consign-mongo    log.row_count     24       1.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
18       ts-consign-price-mongo  log.error_count     24       0.00 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00
19       ts-consign-price-mongo    log.row_count     24       1.85 2024-02-20 18:57:00+00:00 2024-02-20 19:20:00+00:00, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse loaded DataFrames: df_metric, df_trace, df_log (assumed present in kernel from earlier steps)
# Ensure timestamp columns are datetimes in UTC
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Incident window (UTC)
start = pd.to_datetime("2024-02-20 18:54:00", utc=True)
end = pd.to_datetime("2024-02-20 19:24:00", utc=True)

# -------------------------
# 1) metric.csv analysis
# -------------------------
# Compute global P95 per (cmdb_id, kpi_name) across full series
metric_p95 = (
    df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

# Merge global_p95 into metric rows
df_metric_with_p95 = df_metric.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Filter to incident window
df_metric_window = df_metric_with_p95[(df_metric_with_p95['timestamp'] >= start) & (df_metric_with_p95['timestamp'] <= end)]

# For each group in window, find anomalies where value >= global_p95
metric_rows = []
for (cmdb, kpi), g in df_metric_window.groupby(['cmdb_id', 'kpi_name']):
    global_p95 = g['global_p95'].iloc[0]
    count_rows_in_window = len(g)
    anomalies_mask = g['value'] >= global_p95
    count_anomalies = int(anomalies_mask.sum())
    if count_anomalies == 0:
        continue  # only include groups with at least one anomaly
    earliest_anom_ts = g.loc[anomalies_mask, 'timestamp'].min()
    max_value_in_window = float(g['value'].max())
    severity_ratio = (max_value_in_window / global_p95) if global_p95 != 0 else np.inf
    metric_rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': float(global_p95),
        'count_rows_in_window': int(count_rows_in_window),
        'count_anomalies': int(count_anomalies),
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_value_in_window': max_value_in_window,
        'severity_ratio': float(severity_ratio),
    })

metric_anoms = pd.DataFrame(metric_rows)
if not metric_anoms.empty:
    metric_anoms = metric_anoms.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# -------------------------
# 2) trace.csv analysis
# -------------------------
# Compute global P95 per (cmdb_id, trace_name) across full series
trace_p95 = (
    df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'global_p95'})
)

df_trace_with_p95 = df_trace.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')

# Filter to incident window
df_trace_window = df_trace_with_p95[(df_trace_with_p95['timestamp'] >= start) & (df_trace_with_p95['timestamp'] <= end)]

trace_rows = []
for (cmdb, tname), g in df_trace_window.groupby(['cmdb_id', 'trace_name']):
    global_p95 = g['global_p95'].iloc[0]
    count_rows_in_window = len(g)
    anomalies_mask = g['value'] >= global_p95
    count_anomalies = int(anomalies_mask.sum())
    if count_anomalies == 0:
        continue
    earliest_anom_ts = g.loc[anomalies_mask, 'timestamp'].min()
    max_value_in_window = float(g['value'].max())
    severity_ratio = (max_value_in_window / global_p95) if global_p95 != 0 else np.inf
    trace_rows.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'global_p95': float(global_p95),
        'count_rows_in_window': int(count_rows_in_window),
        'count_anomalies': int(count_anomalies),
        'earliest_anomaly_timestamp': earliest_anom_ts,
        'max_value_in_window': max_value_in_window,
        'severity_ratio': float(severity_ratio),
    })

trace_anoms = pd.DataFrame(trace_rows)
if not trace_anoms.empty:
    trace_anoms = trace_anoms.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# -------------------------
# 3) log.csv analysis
# -------------------------
# Compute global P95 and P5 per (cmdb_id, log_name) across full series
log_p95 = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})
log_p5  = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.05).rename(columns={'value': 'global_p5'})

log_thresholds = log_p95.merge(log_p5, on=['cmdb_id', 'log_name'], how='outer')

df_log_with_thr = df_log.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# Filter to incident window
df_log_window = df_log_with_thr[(df_log_with_thr['timestamp'] >= start) & (df_log_with_thr['timestamp'] <= end)]

log_rows = []
for (cmdb, lname), g in df_log_window.groupby(['cmdb_id', 'log_name']):
    global_p95 = g['global_p95'].iloc[0]
    global_p5 = g['global_p5'].iloc[0]
    count_rows_in_window = len(g)
    # Determine anomaly rules
    if lname == 'log.error_count':
        high_mask = g['value'] > global_p95
        low_mask = pd.Series([False]*len(g), index=g.index)
    else:
        # For log.row_count and others: high is >= p95, low is <= p5
        high_mask = g['value'] >= global_p95
        low_mask = g['value'] <= global_p5

    count_high = int(high_mask.sum())
    earliest_high_ts = g.loc[high_mask, 'timestamp'].min() if count_high > 0 else pd.NaT
    count_low = int(low_mask.sum())
    earliest_low_ts = g.loc[low_mask, 'timestamp'].min() if count_low > 0 else pd.NaT
    max_value_in_window = float(g['value'].max())
    min_value_in_window = float(g['value'].min())

    log_rows.append({
        'cmdb_id': cmdb,
        'log_name': lname,
        'global_p95': float(global_p95),
        'global_p5': float(global_p5),
        'count_rows_in_window': int(count_rows_in_window),
        'count_high_anomalies': count_high,
        'earliest_high_anomaly_timestamp': earliest_high_ts,
        'count_low_anomalies': count_low,
        'earliest_low_anomaly_timestamp': earliest_low_ts,
        'max_value_in_window': max_value_in_window,
        'min_value_in_window': min_value_in_window,
    })

log_anoms = pd.DataFrame(log_rows)
if not log_anoms.empty:
    # Sort by total anomalies desc
    log_anoms['total_anomalies'] = log_anoms['count_high_anomalies'] + log_anoms['count_low_anomalies']
    log_anoms = log_anoms.sort_values(['total_anomalies', 'count_rows_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)
    log_anoms = log_anoms.drop(columns=['total_anomalies'])

# Final compact outputs (max 20 rows each)
metric_anoms, trace_anoms, log_anoms

```
Out[2]:
```
Summary of anomaly detection in the incident window [2024-02-20 18:54:00 → 2024-02-20 19:24:00 UTC]:

Metric anomalies (top findings)
- Many (cmdb_id, kpi) groups show a global P95 of 0 which caused every value >= P95 and produced infinite severity_ratio (this is a data/threshold artifact). Examples: many diskio groups (ts-voucher-service, ts-voucher-mysql, ts-avatar-service, ts-ticket-office-service, ts-food-mysql, ts-ui-dashboard) and several error KPIs (ts-seat-service error, ts-food-service error, ts-admin-basic-info error).
- Meaningful nonzero spikes (sorted by severity_ratio):
  - ts-travel-service / diskio: global_p95 ≈ 60,519 ; max in window ≈ 160,844 ; severity ≈ 2.66 (25 rows, 2 anomalous rows)
  - ts-seat-service / cpu: global_p95 ≈ 5.865 ; max ≈ 13.255 ; severity ≈ 2.26 (25 rows, 2 anomalies; earliest anomaly 2024-02-20 19:10:00 UTC)
  - ts-travel-service / latency-50: global_p95 ≈ 0.021 ; max ≈ 0.0409 ; severity ≈ 1.95 (25 rows, 2 anomalies)
  - ts-admin-travel-service / latency-90: global_p95 ≈ 2.255 ; max ≈ 4.293 ; severity ≈ 1.90 (25 rows, 2 anomalies)
  - ts-inside-payment-service / cpu: global_p95 ≈ 0.854 ; max ≈ 1.460 ; severity ≈ 1.71 (25 rows, 2 anomalies)
  - Several other services show moderate CPU/latency/workload spikes with severity ~1.5–1.7.

Trace anomalies (top findings)
- Almost all top trace results are for error_rate metrics whose global P95 is 0. Because P95 = 0, the window rows meet the >= P95 condition and produce infinite severity_ratio. Examples include many trace.to_ts-* and trace.from_ts-* error_rate metrics across services (root, auth, contacts, admin-basic-info, consin, inside-payment, food-service, admin-travel, etc.). These are artifacts of zero-valued P95 rather than large nonzero error values.
- No high non-zero-duration trace anomalies appear at the top of the severity-sorted list (the shown top items are dominated by zero-P95 behavior).

Log anomalies (top findings)
- Most top entries are log.row_count for many *-mongo services. For many of these groups global_p95 = global_p5 = 1.0 and the observed values in-window are mostly 1.0. Because the detection rules used >= for high and <= for low, rows equal to 1.0 get counted as both high and low anomalies — producing count_high = count_low = 24 for several groups (this is a threshold-definition artifact).
- Some groups show real deviation from the baseline:
  - ts-consign-price-mongo, ts-contacts-mongo, ts-order-other-mongo, ts-price-mongo, ts-route-mongo, ts-security-mongo: global_p95 ≈ 1.85–2.00, with some higher values in-window (max values 2–15 for some groups) and a few high-anomaly timestamps around 2024-02-20 19:04:00 UTC.
- log.error_count was not highlighted with meaningful high anomalies in the top results.

Overall interpretation / notes
- A number of flagged “anomalies” are caused by zero or equal thresholds (global P95 or P5 == 0 or P95==P5==1) which makes the simple >= / <= detection rules produce many matches; these should be treated as threshold/data artifacts and inspected further (e.g., check whether the KPI series actually has no variance or whether 0/1 are sentinel/normalized values).
- Outside of threshold artifacts, meaningful spikes during the incident window include:
  - High disk I/O on ts-travel-service (severity ~2.66).
  - CPU and latency spikes on ts-seat-service, ts-travel-service, ts-admin-travel-service, and ts-inside-payment-service (severity ratios ~1.5–2.3).
  - Some mongo-related services show bursts in log.row_count around ~19:04 UTC.
- Recommendation: focus investigation on services with nonzero, high severity_ratio (ts-travel-service diskio; ts-seat-service CPU and latencies; admin-travel and inside-payment CPU/latency), and also review the metric/trace series with zero P95 values to determine if they reflect real steady-zero behavior or telemetry/normalization issues.

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name    global_p95  count_rows_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0            ts-voucher-service      diskio      0.000000                    25               25  2024-02-20 18:57:00+00:00             0.000000             inf
1              ts-voucher-mysql      diskio      0.000000                    25               25  2024-02-20 18:57:00+00:00             0.000000             inf
2             ts-avatar-service      diskio      0.000000                    25               25  2024-02-20 18:57:00+00:00             0.000000             inf
3               ts-seat-service       error      0.000000                    13               13  2024-02-20 19:09:00+00:00             0.000000             inf
4      ts-ticket-office-service      diskio      0.000000                    25               25  2024-02-20 18:57:00+00:00             0.000000             inf
5                 ts-food-mysql      diskio      0.000000                    25               25  2024-02-20 18:57:00+00:00             0.000000             inf
6               ts-food-service       error      0.000000                    10               10  2024-02-20 19:12:00+00:00             0.000000             inf
7               ts-ui-dashboard      diskio      0.000000                    25               25  2024-02-20 18:57:00+00:00             0.000000             inf
8   ts-admin-basic-info-service       error      0.000000                    25               25  2024-02-20 18:57:00+00:00             0.000000             inf
9             ts-travel-service      diskio  60519.186798                    25                2  2024-02-20 18:57:00+00:00        160844.285361        2.657740
10              ts-seat-service         cpu      5.864981                    25                2  2024-02-20 19:10:00+00:00            13.255312        2.260077
11            ts-travel-service  latency-50      0.021022                    25                2  2024-02-20 19:10:00+00:00             0.040908        1.945901
12      ts-admin-travel-service  latency-90      2.255139                    25                2  2024-02-20 19:09:00+00:00             4.292967        1.903637
13    ts-inside-payment-service         cpu      0.854306                    25                2  2024-02-20 19:16:00+00:00             1.460472        1.709541
14           ts-consign-service         cpu      0.522122                    25                2  2024-02-20 19:04:00+00:00             0.872356        1.670788
15           ts-payment-service    workload      0.286617                    25                2  2024-02-20 19:19:00+00:00             0.467000        1.629354
16              ts-seat-service  latency-50      0.046947                    25                2  2024-02-20 19:10:00+00:00             0.074851        1.594366
17          ts-food-map-service         cpu      0.674516                    25                2  2024-02-20 18:59:00+00:00             1.048325        1.554187
18  ts-admin-basic-info-service         cpu      0.767860                    25                2  2024-02-20 19:12:00+00:00             1.191937        1.552284
19              ts-ui-dashboard       error      0.130100                    25                2  2024-02-20 19:09:00+00:00             0.200000        1.537279,              cmdb_id                                       trace_name  global_p95  count_rows_in_window  count_anomalies earliest_anomaly_timestamp  max_value_in_window  severity_ratio
0               root             trace.to_ts-basic-service.error_rate         0.0                     1                1  2024-02-20 18:57:00+00:00                  0.0             inf
1               root              trace.to_ts-auth-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
2    ts-user-service              trace.to_ts-user-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
3               root          trace.to_ts-contacts-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
4               root  trace.to_ts-admin-basic-info-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
5               root           trace.to_ts-consign-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
6               root    trace.to_ts-inside-payment-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
7               root              trace.to_ts-food-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
8               root      trace.to_ts-admin-travel-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
9    ts-seat-service        trace.from_ts-preserve-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
10   ts-seat-service            trace.from_ts-seat-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
11   ts-seat-service                       trace.from_root.error_rate         0.0                     1                1  2024-02-20 18:57:00+00:00                  0.0             inf
12   ts-seat-service       trace.to_ts-order-other-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
13   ts-seat-service          trace.from_ts-travel-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
14   ts-seat-service         trace.from_ts-travel2-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
15  ts-price-service           trace.from_ts-price-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
16   ts-seat-service            trace.to_ts-config-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
17  ts-route-service          trace.from_ts-travel-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
18  ts-price-service             trace.to_ts-price-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf
19  ts-route-service           trace.from_ts-basic-service.error_rate         0.0                    24               24  2024-02-20 18:57:00+00:00                  0.0             inf,                     cmdb_id       log_name  global_p95  global_p5  count_rows_in_window  count_high_anomalies earliest_high_anomaly_timestamp  count_low_anomalies earliest_low_anomaly_timestamp  max_value_in_window  min_value_in_window
0     ts-notification-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   24      2024-02-20 18:57:00+00:00                  1.0                  1.0
1         ts-preserve-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   24      2024-02-20 18:57:00+00:00                  1.0                  1.0
2   ts-preserve-other-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   24      2024-02-20 18:57:00+00:00                  1.0                  1.0
3    ts-ticket-office-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   24      2024-02-20 18:57:00+00:00                  1.0                  1.0
4        ts-assurance-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
5             ts-auth-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
6           ts-config-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
7          ts-consign-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  3.0                  1.0
8         ts-food-map-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
9             ts-food-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
10  ts-inside-payment-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
11           ts-order-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
12         ts-payment-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
13          ts-travel-mongo  log.row_count        1.00        1.0                    24                    24       2024-02-20 18:57:00+00:00                   23      2024-02-20 18:57:00+00:00                  2.0                  1.0
14   ts-consign-price-mongo  log.row_count        1.85        1.0                    24                     2       2024-02-20 19:04:00+00:00                   22      2024-02-20 18:57:00+00:00                  2.0                  1.0
15        ts-contacts-mongo  log.row_count        1.85        1.0                    24                     2       2024-02-20 19:04:00+00:00                   22      2024-02-20 18:57:00+00:00                  4.0                  1.0
16     ts-order-other-mongo  log.row_count        1.85        1.0                    24                     2       2024-02-20 19:04:00+00:00                   22      2024-02-20 18:57:00+00:00                 13.0                  1.0
17           ts-price-mongo  log.row_count        1.85        1.0                    24                     2       2024-02-20 19:04:00+00:00                   22      2024-02-20 18:57:00+00:00                 15.0                  1.0
18           ts-route-mongo  log.row_count        2.00        1.0                    24                     3       2024-02-20 19:04:00+00:00                   21      2024-02-20 18:57:00+00:00                 12.0                  1.0
19        ts-security-mongo  log.row_count        1.85        1.0                    24                     2       2024-02-20 19:04:00+00:00                   22      2024-02-20 18:57:00+00:00                  5.0                  1.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames if present; otherwise load
try:
    df_metric
    df_trace
    df_log
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
    df_log = pd.read_csv('log.csv')
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Ensure timestamps are datetime UTC
df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], utc=True)
df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], utc=True)
df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], utc=True)

# Candidate components
candidates = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']

# Incident window (UTC)
start = pd.to_datetime("2024-02-20 18:54:00", utc=True)
end = pd.to_datetime("2024-02-20 19:24:00", utc=True)

# Helper to group consecutive timestamps (difference > 1 minute breaks sequence)
def group_consecutive(df, ts_col='timestamp', max_gap=pd.Timedelta(minutes=1)):
    if df.shape[0] == 0:
        return pd.Series(dtype=int, index=df.index)
    diffs = df[ts_col].sort_values().diff()
    # First row is start of first group
    grp = (diffs > max_gap).cumsum().fillna(0).astype(int)
    # map back to original index order
    grp = grp.reindex(df.index)
    return grp

# --------------------
# 1) Metric faults
# --------------------
# Compute global P95 across full metric file
metric_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})

# Filter to candidate cmdb_ids and incident window after merging thresholds
df_metric_cand = df_metric[df_metric['cmdb_id'].isin(candidates)].merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')
df_metric_window = df_metric_cand[(df_metric_cand['timestamp'] >= start) & (df_metric_cand['timestamp'] <= end)].copy()

metric_faults_list = []
for (cmdb, kpi), g in df_metric_window.groupby(['cmdb_id', 'kpi_name']):
    g = g.sort_values('timestamp')
    global_p95 = float(g['global_p95'].iloc[0]) if pd.notna(g['global_p95'].iloc[0]) else np.nan
    # Mark anomaly rows where value >= global_p95
    if pd.isna(global_p95):
        continue
    anomalies = g[g['value'] >= global_p95].copy()
    if anomalies.empty:
        continue
    anomalies['grp'] = group_consecutive(anomalies, 'timestamp')
    for gid, ag in anomalies.groupby('grp'):
        fault_len = len(ag)
        fault_start = ag['timestamp'].min()
        fault_end = ag['timestamp'].max()
        max_val = float(ag['value'].max())
        # severity ratio, handle zero p95
        if global_p95 == 0:
            severity = np.inf if max_val > 0 else np.nan
        else:
            severity = max_val / global_p95
        # significance rule
        if (fault_len >= 2) or (max_val >= 1.5 * global_p95):
            metric_faults_list.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'global_p95': global_p95,
                'fault_start_timestamp': fault_start,
                'fault_end_timestamp': fault_end,
                'fault_length_in_samples': int(fault_len),
                'max_value_in_fault': max_val,
                'severity_ratio': float(severity)
            })

metric_faults = pd.DataFrame(metric_faults_list)
if metric_faults.empty:
    metric_faults = metric_faults  # remain empty
else:
    metric_faults = metric_faults.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# --------------------
# 2) Trace faults
# --------------------
# Identify trace rows mentioning any candidate service in trace_name
def trace_mentions_candidate(name):
    for c in candidates:
        if c in name:
            return True
    return False

# Compute global P95 across full trace file
trace_p95 = df_trace.groupby(['cmdb_id', 'trace_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})

# Filter trace rows to those trace_name mentioning candidates
df_trace_cand = df_trace[df_trace['trace_name'].apply(trace_mentions_candidate)].merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')
df_trace_window = df_trace_cand[(df_trace_cand['timestamp'] >= start) & (df_trace_cand['timestamp'] <= end)].copy()

trace_faults_list = []
for (cmdb, tname), g in df_trace_window.groupby(['cmdb_id', 'trace_name']):
    g = g.sort_values('timestamp')
    global_p95 = float(g['global_p95'].iloc[0]) if pd.notna(g['global_p95'].iloc[0]) else np.nan
    if pd.isna(global_p95):
        continue
    anomalies = g[g['value'] >= global_p95].copy()
    if anomalies.empty:
        continue
    anomalies['grp'] = group_consecutive(anomalies, 'timestamp')
    for gid, ag in anomalies.groupby('grp'):
        fault_len = len(ag)
        fault_start = ag['timestamp'].min()
        fault_end = ag['timestamp'].max()
        max_val = float(ag['value'].max())
        if global_p95 == 0:
            severity = np.inf if max_val > 0 else np.nan
        else:
            severity = max_val / global_p95
        if (fault_len >= 2) or (max_val >= 1.5 * global_p95):
            trace_faults_list.append({
                'cmdb_id': cmdb,
                'trace_name': tname,
                'global_p95': global_p95,
                'fault_start_timestamp': fault_start,
                'fault_end_timestamp': fault_end,
                'fault_length_in_samples': int(fault_len),
                'max_value_in_fault': max_val,
                'severity_ratio': float(severity)
            })

trace_faults = pd.DataFrame(trace_faults_list)
if trace_faults.empty:
    trace_faults = trace_faults
else:
    trace_faults = trace_faults.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# --------------------
# 3) Log faults
# --------------------
# Consider only the five services and log names
log_names = ['log.error_count', 'log.row_count']
df_log_cand = df_log[df_log['cmdb_id'].isin(candidates) & df_log['log_name'].isin(log_names)].copy()

# Compute global p95 and p5 across full log file per (cmdb_id, log_name)
log_p95 = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.95).rename(columns={'value': 'global_p95'})
log_p5  = df_log.groupby(['cmdb_id', 'log_name'], as_index=False)['value'].quantile(0.05).rename(columns={'value': 'global_p5'})
log_thr = log_p95.merge(log_p5, on=['cmdb_id', 'log_name'], how='outer')

df_log_cand = df_log_cand.merge(log_thr, on=['cmdb_id', 'log_name'], how='left')
df_log_window = df_log_cand[(df_log_cand['timestamp'] >= start) & (df_log_cand['timestamp'] <= end)].copy()

log_faults_list = []
for (cmdb, lname), g in df_log_window.groupby(['cmdb_id', 'log_name']):
    g = g.sort_values('timestamp')
    global_p95 = float(g['global_p95'].iloc[0]) if pd.notna(g['global_p95'].iloc[0]) else np.nan
    global_p5  = float(g['global_p5'].iloc[0]) if pd.notna(g['global_p5'].iloc[0]) else np.nan
    if pd.isna(global_p95) or pd.isna(global_p5):
        # If thresholds missing, skip (no global data)
        continue
    if lname == 'log.error_count':
        # high anomalies where value > global_p95
        high = g[g['value'] > global_p95].copy()
        # group consecutive high anomalies
        if not high.empty:
            high['grp'] = group_consecutive(high, 'timestamp')
            for gid, ag in high.groupby('grp'):
                fault_len = len(ag)
                start_ts = ag['timestamp'].min()
                end_ts = ag['timestamp'].max()
                max_val = float(ag['value'].max())
                min_val = float(ag['value'].min())
                severity = np.inf if global_p95 == 0 and max_val > 0 else (max_val / global_p95 if global_p95 != 0 else np.nan)
                # include all faults (no additional length threshold specified beyond returning each)
                log_faults_list.append({
                    'cmdb_id': cmdb,
                    'log_name': lname,
                    'anomaly_type': 'high',
                    'global_p95': global_p95,
                    'global_p5': global_p5,
                    'fault_start_timestamp': start_ts,
                    'fault_end_timestamp': end_ts,
                    'fault_length_in_samples': int(fault_len),
                    'max_value_in_fault': max_val,
                    'min_value_in_fault': min_val,
                    'severity_metric': severity
                })
    else:
        # log.row_count: both high (>= p95) and low (<= p5)
        high = g[g['value'] >= global_p95].copy()
        low  = g[g['value'] <= global_p5].copy()
        if not high.empty:
            high['grp'] = group_consecutive(high, 'timestamp')
            for gid, ag in high.groupby('grp'):
                fault_len = len(ag)
                start_ts = ag['timestamp'].min()
                end_ts = ag['timestamp'].max()
                max_val = float(ag['value'].max())
                min_val = float(ag['value'].min())
                severity = np.inf if global_p95 == 0 and max_val > 0 else (max_val / global_p95 if global_p95 != 0 else np.nan)
                log_faults_list.append({
                    'cmdb_id': cmdb,
                    'log_name': lname,
                    'anomaly_type': 'high',
                    'global_p95': global_p95,
                    'global_p5': global_p5,
                    'fault_start_timestamp': start_ts,
                    'fault_end_timestamp': end_ts,
                    'fault_length_in_samples': int(fault_len),
                    'max_value_in_fault': max_val,
                    'min_value_in_fault': min_val,
                    'severity_metric': severity
                })
        if not low.empty:
            low['grp'] = group_consecutive(low, 'timestamp')
            for gid, ag in low.groupby('grp'):
                fault_len = len(ag)
                start_ts = ag['timestamp'].min()
                end_ts = ag['timestamp'].max()
                max_val = float(ag['value'].max())
                min_val = float(ag['value'].min())
                # for low severity, define severity as global_p5 / min_val (larger means deeper drop); handle zeros
                if min_val == 0:
                    severity = np.inf if global_p5 > 0 else np.nan
                else:
                    severity = global_p5 / min_val
                log_faults_list.append({
                    'cmdb_id': cmdb,
                    'log_name': lname,
                    'anomaly_type': 'low',
                    'global_p95': global_p95,
                    'global_p5': global_p5,
                    'fault_start_timestamp': start_ts,
                    'fault_end_timestamp': end_ts,
                    'fault_length_in_samples': int(fault_len),
                    'max_value_in_fault': max_val,
                    'min_value_in_fault': min_val,
                    'severity_metric': severity
                })

log_faults = pd.DataFrame(log_faults_list)
if log_faults.empty:
    # keep empty
    log_faults = log_faults
else:
    # sort by fault length desc then severity_metric desc, return up to 20
    log_faults = log_faults.sort_values(['fault_length_in_samples', 'severity_metric'], ascending=[False, False]).head(20).reset_index(drop=True)
    # drop severity_metric from final display columns (keeps compactness)
    log_faults = log_faults[['cmdb_id','log_name','anomaly_type','global_p95','global_p5','fault_start_timestamp','fault_end_timestamp','fault_length_in_samples','max_value_in_fault','min_value_in_fault']]

# Ensure timestamp columns shown as UTC datetimes (they already are)
metric_faults, trace_faults, log_faults

```
Out[3]:
```
Summary of focused fault identification for the five candidate components (timestamps UTC):

Key finding — primary suspect
- ts-travel-service shows the most significant evidence of a fault:
  - Metric: diskio spike at 2024-02-20 18:57:00 (max 160,844 vs global P95 ≈ 60,519; severity ≈ 2.66). Also CPU and latency-50 spikes around 19:09–19:10 (CPU fault length 2, severity ≈ 1.50; latency-50 single-sample severity ≈ 1.95).
  - Trace: multiple single-sample elevated durations to/from ts-travel-service (notably trace.to_ts-travel-service.duration_p95 and duration_mean) with high severity ratios (examples: food→travel duration_p95 severity ≈ 6.27 at 19:12; admin-travel→travel duration_p95 severity ≈ 4.38 at 19:09; root→travel.duration_p95 severity ≈ 1.51 at 19:09).
  - Log: log.row_count high at 2024-02-20 19:09:00 (value 3166, global_p95 ≈ 2802.1) and several low/high single-sample deviations across the window.

Other notable service observations
- ts-auth-service
  - Metric: two-sample faults (18:59–19:00) for latency-90 (severity ≈ 1.15) and socket (severity ≈ 1.10); small workload blip (19:06–19:07).
  - Trace: root→auth.duration_p95 shows a single-sample increase at 18:59 (severity ≈ 1.57).
  - Log: single-sample log.row_count highs and lows (minor).

- ts-order-service
  - Metric: mem showed a tiny increase (19:20–19:21) with severity ≈ 1.0004 (effectively negligible).
  - Trace: from/to order-service had a two-sample fault at 19:19–19:20 (severity ≈ 1.35) — modest duration increase.
  - Log: both low and high single-sample row_count events around 18:57–19:15 (small in count, worth checking traffic patterns).

- ts-route-service
  - Metric: latency-90 two-sample fault at 19:16–19:17 (severity ≈ 1.02 — marginal). mem very slightly above P95 at 19:20–19:21 (severity ≈ 1.00014).
  - Log: single-sample row_count highs/lows (minor).

- ts-train-service
  - Metric: latency-50 two-sample fault at 19:00–19:01 (severity ≈ 1.0056 — marginal); mem tiny increase at 19:20–19:21 (severity ≈ 1.00006).
  - Trace: two-sample minor duration increases to/from train around 18:59–19:00 (severity ≈ 1.02).

Notes on significance and artifacts
- Many trace and metric “spikes” are single-sample events. We required faults to be length ≥2 OR max ≥1.5×P95; several included single-sample entries because their max exceeded 1.5×P95 (notably ts-travel diskio and some travel-related trace durations).
- Some groups show only marginal increases above P95 (severity ≈ 1.00–1.05) — these are likely noise or minor load variance.
- Log faults are mostly single-sample high/low deviations in row_count; ts-travel-service has the largest log.row_count high at 19:09.

Recommended next steps
1. Prioritize investigation on ts-travel-service (disk I/O spike at 18:57 and correlated CPU/latency and request-volume anomalies around 19:09–19:12).
2. Review ts-travel-service host(s) for disk I/O bottlenecks, thread/connection saturation, and any recent deployments or background jobs at ~18:57–19:12 UTC.
3. Check downstream/upstream traces (services calling or called-by ts-travel-service — e.g., food, admin-travel, seat) for cascading latency increases and increased retries.
4. Triage ts-auth-service and ts-order-service as secondary candidates for connection/latency issues (socket and latency blips), but their severity is lower.
5. Correlate with logs (application and system) and alerts around the highlighted timestamps to confirm root cause (disk I/O and increased traffic to ts-travel-service are the top signals).

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name    global_p95     fault_start_timestamp       fault_end_timestamp  fault_length_in_samples  max_value_in_fault  severity_ratio
0   ts-travel-service      diskio  6.051919e+04 2024-02-20 18:57:00+00:00 2024-02-20 18:57:00+00:00                        1        1.608443e+05        2.657740
1   ts-travel-service  latency-50  2.102240e-02 2024-02-20 19:10:00+00:00 2024-02-20 19:10:00+00:00                        1        4.090752e-02        1.945901
2   ts-travel-service         cpu  4.747153e+00 2024-02-20 19:09:00+00:00 2024-02-20 19:10:00+00:00                        2        7.104663e+00        1.496615
3     ts-auth-service  latency-90  1.137036e+00 2024-02-20 18:59:00+00:00 2024-02-20 19:00:00+00:00                        2        1.311556e+00        1.153487
4     ts-auth-service      socket  1.421333e+01 2024-02-20 18:59:00+00:00 2024-02-20 19:00:00+00:00                        2        1.570000e+01        1.104597
5    ts-route-service  latency-90  8.444546e-03 2024-02-20 19:16:00+00:00 2024-02-20 19:17:00+00:00                        2        8.597283e-03        1.018087
6     ts-auth-service    workload  3.208530e+00 2024-02-20 19:06:00+00:00 2024-02-20 19:07:00+00:00                        2        3.258967e+00        1.015720
7    ts-train-service  latency-50  7.063404e-03 2024-02-20 19:00:00+00:00 2024-02-20 19:01:00+00:00                        2        7.103009e-03        1.005607
8    ts-order-service         mem  2.624508e+08 2024-02-20 19:20:00+00:00 2024-02-20 19:21:00+00:00                        2        2.625592e+08        1.000413
9    ts-route-service         mem  2.622621e+08 2024-02-20 19:20:00+00:00 2024-02-20 19:21:00+00:00                        2        2.622997e+08        1.000143
10   ts-train-service         mem  2.582609e+08 2024-02-20 19:20:00+00:00 2024-02-20 19:21:00+00:00                        2        2.582774e+08        1.000064,                       cmdb_id                                  trace_name  global_p95     fault_start_timestamp       fault_end_timestamp  fault_length_in_samples  max_value_in_fault  severity_ratio
0             ts-food-service     trace.to_ts-travel-service.duration_p95    0.543400 2024-02-20 19:12:00+00:00 2024-02-20 19:12:00+00:00                        1            3.408307        6.272184
1             ts-food-service    trace.to_ts-travel-service.duration_mean    0.120244 2024-02-20 19:12:00+00:00 2024-02-20 19:12:00+00:00                        1            0.639662        5.319718
2     ts-admin-travel-service     trace.to_ts-travel-service.duration_p95    1.240552 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1            5.436433        4.382271
3         ts-preserve-service      trace.to_ts-order-service.duration_p95    0.019444 2024-02-20 18:57:00+00:00 2024-02-20 18:57:00+00:00                        1            0.072968        3.752748
4     ts-admin-travel-service    trace.to_ts-travel-service.duration_mean    0.527614 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1            1.636612        3.101911
5             ts-seat-service   trace.from_ts-travel-service.duration_p95    0.732299 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1            2.033019        2.776214
6   ts-inside-payment-service      trace.to_ts-order-service.duration_p95    0.008916 2024-02-20 18:57:00+00:00 2024-02-20 18:57:00+00:00                        1            0.024029        2.694967
7             ts-seat-service    trace.to_ts-travel-service.duration_mean    0.069701 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1            0.187672        2.692537
8             ts-seat-service  trace.from_ts-travel-service.duration_mean    0.228658 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1            0.489051        2.138789
9         ts-preserve-service     trace.to_ts-order-service.duration_mean    0.015891 2024-02-20 18:57:00+00:00 2024-02-20 18:57:00+00:00                        1            0.030103        1.894371
10            ts-seat-service     trace.to_ts-travel-service.duration_p95    0.266036 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1            0.472126        1.774669
11                       root       trace.to_ts-auth-service.duration_p95    1.107602 2024-02-20 18:59:00+00:00 2024-02-20 18:59:00+00:00                        1            1.743481        1.574104
12  ts-inside-payment-service     trace.to_ts-order-service.duration_mean    0.005214 2024-02-20 18:57:00+00:00 2024-02-20 18:57:00+00:00                        1            0.008089        1.551451
13                       root     trace.to_ts-travel-service.duration_p95    5.094279 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1            7.693809        1.510284
14           ts-order-service    trace.from_ts-order-service.duration_p95    0.010392 2024-02-20 19:19:00+00:00 2024-02-20 19:20:00+00:00                        2            0.014011        1.348211
15           ts-order-service      trace.to_ts-order-service.duration_p95    0.010392 2024-02-20 19:19:00+00:00 2024-02-20 19:20:00+00:00                        2            0.014011        1.348211
16           ts-basic-service     trace.to_ts-train-service.duration_mean    0.004884 2024-02-20 18:59:00+00:00 2024-02-20 19:00:00+00:00                        2            0.005044        1.032801
17            ts-food-service        trace.to_ts-travel-service.row_count   32.000000 2024-02-20 19:00:00+00:00 2024-02-20 19:01:00+00:00                        2           33.000000        1.031250
18           ts-train-service     trace.to_ts-train-service.duration_mean    0.001185 2024-02-20 18:59:00+00:00 2024-02-20 19:00:00+00:00                        2            0.001211        1.022291
19           ts-train-service   trace.from_ts-train-service.duration_mean    0.001185 2024-02-20 18:59:00+00:00 2024-02-20 19:00:00+00:00                        2            0.001211        1.022291,               cmdb_id       log_name anomaly_type  global_p95  global_p5     fault_start_timestamp       fault_end_timestamp  fault_length_in_samples  max_value_in_fault  min_value_in_fault
0   ts-travel-service  log.row_count         high      2802.1    1414.95 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1              3166.0              3166.0
1    ts-order-service  log.row_count          low       545.7     301.05 2024-02-20 19:12:00+00:00 2024-02-20 19:12:00+00:00                        1               270.0               270.0
2   ts-travel-service  log.row_count          low      2802.1    1414.95 2024-02-20 18:57:00+00:00 2024-02-20 18:57:00+00:00                        1              1320.0              1320.0
3    ts-route-service  log.row_count          low      1215.7    1007.05 2024-02-20 19:12:00+00:00 2024-02-20 19:12:00+00:00                        1               971.0               971.0
4   ts-travel-service  log.row_count          low      2802.1    1414.95 2024-02-20 19:07:00+00:00 2024-02-20 19:07:00+00:00                        1              1386.0              1386.0
5     ts-auth-service  log.row_count          low       392.8     342.60 2024-02-20 19:11:00+00:00 2024-02-20 19:11:00+00:00                        1               336.0               336.0
6    ts-order-service  log.row_count         high       545.7     301.05 2024-02-20 19:15:00+00:00 2024-02-20 19:15:00+00:00                        1               555.0               555.0
7   ts-travel-service  log.row_count         high      2802.1    1414.95 2024-02-20 19:12:00+00:00 2024-02-20 19:12:00+00:00                        1              2842.0              2842.0
8    ts-order-service  log.row_count          low       545.7     301.05 2024-02-20 18:57:00+00:00 2024-02-20 18:57:00+00:00                        1               297.0               297.0
9     ts-auth-service  log.row_count         high       392.8     342.60 2024-02-20 19:06:00+00:00 2024-02-20 19:06:00+00:00                        1               398.0               398.0
10   ts-route-service  log.row_count         high      1215.7    1007.05 2024-02-20 19:02:00+00:00 2024-02-20 19:02:00+00:00                        1              1228.0              1228.0
11   ts-route-service  log.row_count          low      1215.7    1007.05 2024-02-20 19:09:00+00:00 2024-02-20 19:09:00+00:00                        1              1003.0              1003.0
12    ts-auth-service  log.row_count         high       392.8     342.60 2024-02-20 19:08:00+00:00 2024-02-20 19:08:00+00:00                        1               394.0               394.0
13   ts-route-service  log.row_count         high      1215.7    1007.05 2024-02-20 19:06:00+00:00 2024-02-20 19:06:00+00:00                        1              1219.0              1219.0
14    ts-auth-service  log.row_count          low       392.8     342.60 2024-02-20 19:18:00+00:00 2024-02-20 19:18:00+00:00                        1               342.0               342.0
15   ts-order-service  log.row_count         high       545.7     301.05 2024-02-20 19:04:00+00:00 2024-02-20 19:04:00+00:00                        1               546.0               546.0)```
```